# Train Mask2Former (nanostars) on Kaggle

Runs the repo end-to-end on a Kaggle GPU notebook with no source edits. See
`kaggle/README.md` for dataset setup.

**Before running:** attach the data (and code) dataset(s); set **Accelerator: GPU P100**
and **Internet: On**; edit the config cell. Keep `SMOKE_TEST = True` for the first run.

**To leave it running for hours:** use **Save Version → Save & Run All (Commit)** — that
executes headless (browser can be closed) for up to 12h and saves outputs to the version's
**Output** tab. The interactive session is only for the smoke test.

In [ ]:
# ---- config (edit me) -------------------------------------------------
USE_DENOISING      = True    # overlap-handling branch; False = plain Mask2Former
IMG_SIZE           = 1024    # model input resolution
BATCH_SIZE         = 2       # P100/T4 16GB: try 2 at 1024, drop to 1 on CUDA OOM
EPOCHS             = 20
SAMPLES_PER_EPOCH  = 300
SMOKE_TEST         = True    # first run: 1 epoch / 8 samples to prove it works
MODEL              = "facebook/mask2former-swin-tiny-coco-instance"

# Optional explicit input paths (leave "" to auto-detect under /kaggle/input).
# Use these if you split code and data into separate datasets (recommended:
# keeps the 4GB image dataset fixed while code updates are tiny re-uploads).
CODE_DIR    = ""   # folder containing src/   e.g. /kaggle/input/starmeter-code
DATA_DIR_IN = ""   # folder containing annotations/ and images/

In [ ]:
# ---- dependencies -----------------------------------------------------
# transformers is pinned: src/m2f_denoise.py hooks Mask2Former decoder internals
# validated against 4.41.0 (plain Mask2Former works on newer too).
!pip install -q "transformers==4.41.0" "scipy>=1.10"
import transformers; print("transformers", transformers.__version__)

In [ ]:
# ---- resolve paths + copy code to a writable dir ----------------------
import glob, os, shutil
from pathlib import Path

def _find(pattern):
    return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

if CODE_DIR:
    SRC_REPO = Path(CODE_DIR)
else:
    hits = _find("src/m2f_train.py")
    assert hits, "src/m2f_train.py not found under /kaggle/input — set CODE_DIR or attach the code dataset."
    SRC_REPO = Path(hits[0]).parents[1]
assert (SRC_REPO / "src" / "m2f_train.py").exists(), f"no src/m2f_train.py in {SRC_REPO}"

if DATA_DIR_IN:
    DATA_DIR = Path(DATA_DIR_IN)
else:
    hits = _find("annotations/train.json")
    assert hits, "annotations/train.json not found under /kaggle/input — set DATA_DIR_IN or attach the data dataset."
    DATA_DIR = Path(hits[0]).parents[1]

IMAGES = DATA_DIR / "images"
COCO   = DATA_DIR / "annotations" / "train.json"
EVAL   = DATA_DIR / "annotations" / "eval.json"

WORK = Path("/kaggle/working/starMeter")
if WORK.exists():
    shutil.rmtree(WORK)
shutil.copytree(SRC_REPO / "src", WORK / "src")     # only need the code
os.chdir(WORK)                                       # so `python src/...` resolves + writes pyc

OUT  = Path("/kaggle/working/checkpoints/mask2former-nanostar")
PRED = Path("/kaggle/working/predictions_m2f.json")

print("code   :", SRC_REPO)
print("images :", IMAGES, "(", len(list(IMAGES.glob('*'))), "files )")
print("train  :", COCO)
print("eval   :", EVAL if EVAL.exists() else "(none)")
print("workdir:", WORK)

In [ ]:
# ---- sanity: GPU ------------------------------------------------------
import torch
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(no GPU — enable the accelerator)")

In [ ]:
# ---- train ------------------------------------------------------------
# Checkpoints are saved AFTER EVERY EPOCH to OUT, so a long unattended run that
# times out still leaves the latest usable model.
dn  = "--dn --lambda-p 0.2" if USE_DENOISING else ""
ep  = 1 if SMOKE_TEST else EPOCHS
spe = 8 if SMOKE_TEST else SAMPLES_PER_EPOCH
cmd = (
    f"python src/m2f_train.py "
    f"--model {MODEL} --coco {COCO} --images-dir {IMAGES} --out {OUT} "
    f"--size {IMG_SIZE} --batch-size {BATCH_SIZE} --epochs {ep} "
    f"--samples-per-epoch {spe} --num-workers 2 {dn}"
)
print(cmd, "\n")
!{cmd}

In [ ]:
# ---- inference on eval.json ------------------------------------------
if EVAL.exists():
    cmd = (
        f"python src/m2f_pipeline.py "
        f"--ckpt {OUT} --input-json {EVAL} --images-dir {IMAGES} "
        f"--out {PRED} --size {IMG_SIZE}"
    )
    print(cmd, "\n")
    !{cmd}
else:
    print("no eval.json — skipping inference")

## Outputs (in `/kaggle/working`)
- `checkpoints/mask2former-nanostar/` — plain Mask2Former checkpoint (dn params stripped), re-saved each epoch
- `predictions_m2f.json` — COCO predictions over `eval.json`

Once the smoke run succeeds, set `SMOKE_TEST = False` and use **Save Version → Save & Run All**
to run the full job headless. Outputs appear on the finished version's **Output** tab.